Basic Imports

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Load the dataset

In [ ]:
data = pd.read_csv("insurance.csv")

data.head()

In [ ]:
data.shape

In [ ]:
data.info()

Check the feature correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

corr = data.corr()
sns.heatmap(corr , annot = True , ax=ax)

Label Encode Object Types

In [ ]:
d_types = dict(data.dtypes)
for name , type_ in d_types.items():
    if str(type_) == 'object':
        print(f"<======== {name} ===========>")
        print(data[name].value_counts())
        print()

In [ ]:
from sklearn.preprocessing import LabelEncoder

for name , type_ in d_types.items():
    if str(type_) == 'object':
        Le = LabelEncoder()
        data[name] = Le.fit_transform(data[name])

Check info after Label Encoding

In [ ]:
data.info()

One hot Encoding 

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder()
part = onehotencoder.fit_transform(data['region'].values.reshape(-1,1)).toarray()

values = dict(data["region"].value_counts())

for e , (val , _) in enumerate(values.items()):
    data["region_" + str(val)] = part[:,e]

data = data.drop(["region"] , axis = 1)

data.head()

In [ ]:
data.info()

Handle Skewness in Predictive column

In [ ]:
Original_Y = data["expenses"].values.copy()

In [ ]:
Original_Y

In [ ]:
print("Skewness in Column : Expenses " , data["expenses"].skew())

plt.hist(data["expenses"])
plt.show()

In [ ]:
col_log = np.log(data["expenses"])
print("Skewness in Column : Log Expenses " , col_log.skew())

plt.hist(col_log)
plt.show()

In [ ]:
col_sqrt = np.sqrt(data["expenses"])

print("Skewness in Column : Sqrt Expenses " ,col_sqrt.skew())

plt.hist(col_sqrt)
plt.show()

In [ ]:
from scipy import stats 

col_cox , lam = stats.boxcox(data["expenses"])[0:2]
print("Skewness in Column : Sqrt Expenses " ,pd.Series(col_cox).skew())

plt.hist(col_cox)
plt.show()

In [ ]:
data["expenses"] = col_cox

Make Features and Targets

In [ ]:
remaining_columns = list(data.columns)
remaining_columns.remove("expenses")

In [ ]:
X = data[remaining_columns].values 
Y = data['expenses'].values

In [ ]:
from sklearn.model_selection import KFold 
from sklearn.preprocessing import StandardScaler

Scaler = StandardScaler()
X = Scaler.fit_transform(X)

In [ ]:
# check whether data is standardized or not 
# mean should be 1 

plt.ylim(-1,1)

means = []
for i in range(X.shape[1]):
    means.append(np.mean(X[:,i]))
plt.plot(means , scaley=False)

In [ ]:
# Check variances 

plt.ylim(0,2)

vars = []
for i in range(X.shape[1]):
    vars.append(np.var(X[:,i]))
plt.plot(vars)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA 

pca = PCA(n_components = 7)
X = pca.fit_transform(X)

pca.explained_variance_ratio_.cumsum()

Defining Metrics 

In [ ]:
def rmse_score(y_test , y_pred):
    value = (1/len(y_test))*np.sum((y_test - y_pred)**2)
    return np.sqrt(value)

def r2_score(y_test , y_pred):
    ssr = (1/len(y_test))*np.sum((y_test - y_pred)**2)
    sst = (1/len(y_test))*np.sum((y_test - np.mean(y_test))**2)
    return (1 - (ssr/sst))

def mae(y_test , y_pred):
    return (1/len(y_test))*np.sum(np.abs(y_test - y_pred))

def adj_r2_score(y_test , y_pred , n_features):
    numerator = (1-r2_score(y_test , y_pred))*(len(y_test) - 1)
    denominator = len(y_test) - n_features - 1
    return 1 - (numerator/denominator)

In [ ]:
k_fold = KFold(n_splits=5)

# Plotting Root mean squared error 
rmse_scores = []
r2_scores = []
mae_scores = []
r2_adj_scores = []

for train_idx , test_idx in k_fold.split(X):
    Xtrain = X[train_idx]
    Ytrain = Y[train_idx]

    Xtest = X[test_idx]
    Ytest = Y[test_idx]

    model = LinearRegression()
    model.fit(Xtrain , Ytrain)

    Ypred = model.predict(Xtest)
    rmse_scores.append(rmse_score(Ytest , Ypred))
    r2_scores.append(r2_score(Ytest , Ypred))
    mae_scores.append(mae(Ytest , Ypred))
    r2_adj_scores.append(adj_r2_score(Ytest , Ypred , Xtest.shape[1]))

print(" Average RMSE " , np.mean(rmse_scores))
plt.plot(rmse_scores)
plt.plot([np.mean(rmse_scores)]*len(rmse_scores))
plt.title(" RMSE ")
plt.show()

print(" Average MAE " , np.mean(mae_scores))
plt.plot(mae_scores)
plt.plot([np.mean(mae_scores)]*len(mae_scores))
plt.title(" MAE ")
plt.show()

print(" Average R square " , np.mean(r2_scores))
plt.plot(r2_scores)
plt.plot([np.mean(r2_scores)]*len(r2_scores))
plt.title(" R square ")
plt.show()

print(" Average Adj R square " , np.mean(r2_adj_scores))
plt.plot(r2_adj_scores)
plt.plot([np.mean(r2_adj_scores)]*len(r2_adj_scores))
plt.title(" Adj R square ")
plt.show()

Can we Bring back the data?

In [ ]:
from scipy.special import inv_boxcox

Real_data = inv_boxcox(Y , lam)

In [ ]:
Real_data[:10]

In [ ]:
Original_Y[:10]